# Annotator Agreement

Inter-annotator agreement across the three tasks.

- **Setting** — 4 Likert dimensions (1–5): concreteness, temporal grounding, spatial grounding, sensory
- **Agency** — 5 Likert dimensions (1–5): focalization, emotion, cognition, change of state, conflict
- **Event Relation** — span event labels (binary), temporal order (nominal), causality (nominal)

Ordinal dimensions report exact match, within-1, MAE, Krippendorff's α (interval)
and ICC(2,1). Nominal dimensions report exact match, Cohen's κ, α (nominal) and F1.
Rows with fewer than 3 shared instances show an em dash.

The metrics live in `agreement.py` and the tables in `agreement_report.py`, so this
notebook is the analysis rather than the machinery.

In [ ]:
from nb_utils import setup_plots, load, DIMENSIONS, TASK_COLOR
from agreement import TASK_LABEL, er_metric_pairs
from agreement_report import AgreementReport

plt = setup_plots()

# Loads every task's annotations once; all views below are rendered from it.
rep = AgreementReport()

for task in rep.tasks:
    counts = {a: len(d) for a, d in rep.all_data[task].items()}
    print(f'{TASK_LABEL[task]:15s} {counts}')

## Headline numbers

One row per task and annotator pair, averaged over that task's dimensions.

In [ ]:
rep.summary()

## Every pair, every dimension

In [ ]:
rep.table()

## Per task

Each of these stands alone — run just the one you care about.

In [ ]:
rep.table('setting')

In [ ]:
rep.table('agency')

In [ ]:
rep.table('event_relation')

## Where they disagreed

Cards are sorted worst-first by the largest disagreement on the instance. The
passage is shown with the two assigned spans highlighted, followed by every
annotator's ratings. Adjust `min_diff`, `max_show`, or pass `feature=` to focus
on a single dimension.

In [ ]:
rep.disagreements('setting', ann1='tejo9855', ann2='roda9210', feature='concreteness')

In [ ]:
rep.disagreements('agency')

In [ ]:
rep.disagreements('event_relation', ann1='tejo9855', ann2='maria')

## Rating distributions per annotator

How each person used the scale — useful for spotting calibration differences
that the agreement numbers alone would not explain.

In [ ]:
for task in ('setting', 'agency'):
    dims = DIMENSIONS[task]
    annotators = [a for a in rep.annotators[task] if rep.all_data[task].get(a)]
    if not annotators:
        continue

    fig, axes = plt.subplots(len(annotators), len(dims),
                             figsize=(3.2 * len(dims), 2.8 * len(annotators)),
                             squeeze=False)
    fig.suptitle(f'{TASK_LABEL[task]} — rating distributions',
                 fontsize=13, fontweight='bold', y=1.01)

    for ai, ann in enumerate(annotators):
        records = rep.all_data[task][ann]
        for di, dim in enumerate(dims):
            ax = axes[ai][di]
            vals = [r.get(dim) for r in records.values() if r and r.get(dim) is not None]
            ax.bar(range(1, 6), [vals.count(v) for v in range(1, 6)],
                   color=TASK_COLOR[task], alpha=0.75, edgecolor='white')
            ax.set_xticks(range(1, 6))
            ax.set_xlim(0.3, 5.7)
            ax.set_title(dim.replace(f'{task}_', '').replace('_', ' '), fontsize=9)
            if di == 0:
                ax.set_ylabel(ann, fontsize=8)
            if ai == len(annotators) - 1:
                ax.set_xlabel('rating', fontsize=8)
            ax.tick_params(labelsize=8)

    plt.tight_layout()
    plt.show()

## Event relation — per-instance detail

Every shared instance for each pair, with agreeing cells shaded green.

In [ ]:
from itertools import combinations
from IPython.display import display, HTML

yn = lambda d, key: 'Y' if d.get(key) else 'N'
GREEN = 'background:#d4edda'

er_anns = [a for a in rep.annotators['event_relation'] if rep.all_data['event_relation'].get(a)]

for ann1, ann2 in combinations(er_anns, 2):
    er1, er2 = rep.all_data['event_relation'][ann1], rep.all_data['event_relation'][ann2]
    shared = set(er1) & set(er2)

    body = ''
    for inst_id in rep.order:
        if inst_id not in shared:
            continue
        a1, a2 = er1.get(inst_id), er2.get(inst_id)
        if a1 is None or a2 is None:
            continue
        sp1 = bool(a1.get('span1_is_event')) == bool(a2.get('span1_is_event'))
        sp2 = bool(a1.get('span2_is_event')) == bool(a2.get('span2_is_event'))
        t1, t2 = a1.get('temporal_order'), a2.get('temporal_order')
        c1, c2 = a1.get('causality_rating'), a2.get('causality_rating')
        both_t = t1 is not None and t2 is not None
        both_c = c1 is not None and c2 is not None
        body += (
            '<tr style="border-top:1px solid #eee">'
            f'<td style="padding:4px 10px;font-size:0.8em;color:#888">{inst_id}</td>'
            f'<td style="padding:4px 10px;text-align:center;{GREEN if sp1 else ""}">'
            f'{yn(a1,"span1_is_event")} / {yn(a2,"span1_is_event")}</td>'
            f'<td style="padding:4px 10px;text-align:center;{GREEN if sp2 else ""}">'
            f'{yn(a1,"span2_is_event")} / {yn(a2,"span2_is_event")}</td>'
            f'<td style="padding:4px 10px;text-align:center;{GREEN if both_t and t1 == t2 else ""}">'
            f'{f"{t1} / {t2}" if both_t else "-"}</td>'
            f'<td style="padding:4px 10px;text-align:center;{GREEN if both_c and c1 == c2 else ""}">'
            f'{f"{c1} / {c2}" if both_c else "-"}</td></tr>'
        )

    header = ('<tr style="background:#f0f0f0">'
              '<th style="padding:4px 10px;text-align:left">Instance</th>'
              '<th style="padding:4px 10px;color:#4e9af1">Span 1 event</th>'
              '<th style="padding:4px 10px;color:#f4a432">Span 2 event</th>'
              '<th style="padding:4px 10px">Temporal order</th>'
              '<th style="padding:4px 10px">Causality</th></tr>')

    display(HTML(
        '<div style="font-family:sans-serif;margin-bottom:20px">'
        f'<b>{ann1}</b> vs <b>{ann2}</b> — {len(shared)} shared instances '
        f'<span style="color:#888;font-size:0.85em">(cells read {ann1} / {ann2})</span>'
        '<table style="border-collapse:collapse;font-size:0.9em;margin-top:8px">'
        f'<thead>{header}</thead><tbody>{body}</tbody></table></div>'
    ))

## Event relation — confusion matrices

Rows are the adjudicator, columns the second annotator. Only instances where
both gave a value are counted.

In [ ]:
import numpy as np

ANN_GOLD, ANN_PRED = 'tejo9855', 'maria'
er_gold = rep.all_data['event_relation'].get(ANN_GOLD, {})
er_pred = rep.all_data['event_relation'].get(ANN_PRED, {})
shared  = set(er_gold) & set(er_pred)


def confusion(gold_vals, pred_vals, labels):
    idx = {l: i for i, l in enumerate(labels)}
    mat = np.zeros((len(labels), len(labels)), dtype=int)
    for g, p in zip(gold_vals, pred_vals):
        if g in idx and p in idx:
            mat[idx[g]][idx[p]] += 1
    return mat


def plot_cm(ax, mat, labels, title, n_pairs):
    im = ax.imshow(mat, cmap='Blues')
    ticks = [str(l) for l in labels]
    ax.set_xticks(range(len(labels)));  ax.set_xticklabels(ticks, rotation=35, ha='right', fontsize=8)
    ax.set_yticks(range(len(labels)));  ax.set_yticklabels(ticks, fontsize=8)
    ax.set_xlabel(ANN_PRED, fontsize=9); ax.set_ylabel(ANN_GOLD, fontsize=9)
    ax.set_title(f'{title}\n(n={n_pairs})', fontsize=10)
    ax.grid(False)
    thresh = mat.max() / 2
    for i in range(len(labels)):
        for j in range(len(labels)):
            ax.text(j, i, str(mat[i, j]), ha='center', va='center', fontsize=9,
                    color='white' if mat[i, j] > thresh else 'black')
    plt.colorbar(im, ax=ax, shrink=0.75)


NOMINAL = [
    ('Span 1 is event', 'span1_is_event',   [True, False], bool),
    ('Span 2 is event', 'span2_is_event',   [True, False], bool),
    ('Temporal order',  'temporal_order',   ['span1_first', 'span2_first', 'simultaneous',
                                             'same_event', 'too_hard_to_tell'], lambda v: v),
    ('Causality',       'causality_rating', ['direct_cause', 'enables', 'not_related'], lambda v: v),
]

fig, axes = plt.subplots(1, 4, figsize=(22, 5))
fig.suptitle(f'Event relation — nominal confusion matrices '
             f'({ANN_GOLD} = rows, {ANN_PRED} = columns)',
             fontsize=12, fontweight='bold', y=1.02)

for ax, (title, key, raw_labels, transform) in zip(axes, NOMINAL):
    gold_vals, pred_vals = [], []
    for inst_id in rep.order:
        if inst_id not in shared:
            continue
        g, p = er_gold.get(inst_id), er_pred.get(inst_id)
        if g is None or p is None:
            continue
        gv = transform(g[key]) if g.get(key) is not None else None
        pv = transform(p[key]) if p.get(key) is not None else None
        if gv is not None and pv is not None:
            gold_vals.append(gv); pred_vals.append(pv)
    present = set(gold_vals) | set(pred_vals)
    labels  = [l for l in raw_labels if l in present]
    plot_cm(ax, confusion(gold_vals, pred_vals, labels), labels, title, len(gold_vals))

plt.tight_layout()
plt.show()

# The binary views reuse the same pair collection the metric tables use, so the
# matrices and the reported κ/F1 can never drift apart.
binary_pairs = er_metric_pairs(er_gold, er_pred, rep.order)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
fig.suptitle(f'Event relation — binary confusion matrices '
             f'({ANN_GOLD} = rows, {ANN_PRED} = columns)',
             fontsize=12, fontweight='bold', y=1.02)

for ax, (label, title) in zip(axes, [
    ('Temporal order (directional / not)', 'Temporal order\n(1=directional, 0=other)'),
    ('Causality (causal / not)',           'Causality\n(1=causal, 0=not related)'),
]):
    pairs = binary_pairs[label]
    gold_vals = [g for g, _ in pairs]
    pred_vals = [p for _, p in pairs]
    plot_cm(ax, confusion(gold_vals, pred_vals, [1, 0]), [1, 0], title, len(pairs))

plt.tight_layout()
plt.show()

## Annotation drift

Whether ratings shift as an annotator works through the queue.
`annotation_order_{annotator}` records where in their queue each instance fell.
Change `DRIFT_TASK` / `DRIFT_ANNOTATOR` to look at anyone; Likert tasks only.

In [ ]:
DRIFT_TASK      = 'agency'      # 'setting' or 'agency'
DRIFT_ANNOTATOR = 'tejo9855'
DRIFT_BIN_SIZE  = 10            # instances per bin

assert DRIFT_TASK in ('setting', 'agency'), 'drift is defined for the Likert tasks only'

dims      = DIMENSIONS[DRIFT_TASK]
order_col = f'annotation_order_{DRIFT_ANNOTATOR}'

seq = (load(DRIFT_TASK)[[order_col] + [f'{d}_{DRIFT_ANNOTATOR}' for d in dims]]
       .dropna(subset=[order_col])
       .rename(columns={order_col: 'annotation_order',
                        **{f'{d}_{DRIFT_ANNOTATOR}': d for d in dims}})
       .sort_values('annotation_order')
       .reset_index(drop=True))

SCALE_MIN, SCALE_MAX = 1, 5     # all nine Likert dimensions are 1-5

seq['bin'] = (seq['annotation_order'] - 1) // DRIFT_BIN_SIZE
bin_means   = seq.groupby('bin')[dims].mean()
bin_counts  = seq.groupby('bin').size()
bin_centers = bin_means.index * DRIFT_BIN_SIZE + DRIFT_BIN_SIZE / 2

fig, ax = plt.subplots(figsize=(11, 5))
for i, dim in enumerate(dims):
    ax.plot(bin_centers, bin_means[dim], marker='o', linewidth=2,
            label=dim.replace(f'{DRIFT_TASK}_', '').replace('_', ' ').title(),
            color=plt.cm.tab10.colors[i])

ax.axhline((SCALE_MIN + SCALE_MAX) / 2, color='gray', linestyle='--',
           linewidth=0.8, alpha=0.5, label='scale midpoint')
ax.set_ylim(SCALE_MIN - 0.2, SCALE_MAX + 0.2)
ax.set_xlabel(f'Annotation order (center of {DRIFT_BIN_SIZE}-instance bin)')
ax.set_ylabel('Mean rating')
ax.set_title(f'Annotation drift — {TASK_LABEL[DRIFT_TASK]} / {DRIFT_ANNOTATOR}'
             f'  (bin size = {DRIFT_BIN_SIZE}, n = {len(seq)})')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
for bc, bn in zip(bin_centers, bin_counts):
    ax.text(bc, SCALE_MIN - 0.13, f'n={bn}', ha='center', va='top', fontsize=7, color='#888')

plt.tight_layout()
plt.show()